# COMP34812 — Authorship Verification (Track A)
## Approach A: Traditional ML — TF-IDF + Stylometric Features + Compression Distance

**Runtime:** CPU only — ~3-5 minutes total

### Architecture & Creative Elements:

1. **TF-IDF Character N-grams (3-5 grams)** — captures subword stylistic
   signatures (spelling habits, word endings, punctuation patterns)

2. **16 Handcrafted Stylometric Features** per text — vocabulary richness
   (TTR, Yule's K, hapax legomena), function word ratio, punctuation rates,
   sentence structure, contraction usage

3. **Normalized Compression Distance (NCD)** — information-theoretic similarity
   measure inspired by Jiang et al. (ACL 2023) "Low-Resource Text Classification:
   A Parameter-Free Classification Method with Compressors". Intuition: if two
   texts share stylistic patterns, their concatenation compresses better than
   unrelated texts. NCD captures character-level regularities, repetition habits,
   and structural patterns that TF-IDF misses.

4. **Pairwise Difference & Product Features** — absolute difference and
   element-wise product between stylometric vectors

5. **TF-IDF Cosine Similarity** — explicit textual overlap signal

6. **Ensemble of SVM + Logistic Regression** with probability averaging

7. **Threshold Optimisation** on dev set

---
**Target:** ~72-78% F1

## 0. Setup

In [1]:
!pip install -q scikit-learn pandas numpy

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Imports & Config

In [3]:
import os
import re
import gzip
import time
import string
import json
import pickle
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix

warnings.filterwarnings("ignore")

CONFIG = {
    "seed": 42,

    # TF-IDF
    "tfidf_max_features": 50000,
    "tfidf_ngram_range": (3, 5),
    "tfidf_analyzer": "char_wb",

    # Paths — UPDATE THESE
    "train_path": "/content/train.csv",
    "dev_path": "/content/dev.csv",
    "output_dir": "/content/drive/MyDrive/models_approach_a",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
print("Config loaded.")

Config loaded.


## 2. Load Data

In [4]:
def load_data(path):
    df = pd.read_csv(path)
    df["text_1"] = df["text_1"].astype(str).str.strip()
    df["text_2"] = df["text_2"].astype(str).str.strip()
    if "label" in df.columns:
        df["label"] = df["label"].astype(int)
    return df

train_df = load_data(CONFIG["train_path"])
dev_df = load_data(CONFIG["dev_path"])

print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,}")
print(f"Labels — Train: {train_df['label'].value_counts().to_dict()}")
print(f"Labels — Dev:   {dev_df['label'].value_counts().to_dict()}")

Train: 27,643 | Dev: 5,993
Labels — Train: {0: 13950, 1: 13693}
Labels — Dev:   {1: 3056, 0: 2937}


## 3. Stylometric Feature Extraction

16 handcrafted features capturing writing style independent of topic.

In [5]:
FUNCTION_WORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "need", "dare", "ought",
    "used", "to", "of", "in", "for", "on", "with", "at", "by", "from",
    "as", "into", "through", "during", "before", "after", "above", "below",
    "between", "out", "off", "over", "under", "again", "further", "then",
    "once", "and", "but", "or", "nor", "not", "so", "yet", "both",
    "either", "neither", "each", "every", "all", "any", "few", "more",
    "most", "other", "some", "such", "no", "only", "own", "same", "than",
    "too", "very", "just", "because", "if", "when", "while", "although",
    "i", "me", "my", "we", "us", "our", "you", "your", "he", "him",
    "his", "she", "her", "it", "its", "they", "them", "their", "this",
    "that", "these", "those", "what", "which", "who", "whom",
}


def extract_stylometric_features(text):
    """Extract 16 writing-style features from raw text."""
    chars = len(text)
    words = text.split()
    n_words = max(len(words), 1)
    sentences = re.split(r'[.!?]+', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    n_sentences = max(len(sentences), 1)
    words_lower = [w.lower().strip(string.punctuation) for w in words]
    words_lower = [w for w in words_lower if w]
    n_words_clean = max(len(words_lower), 1)

    avg_word_len = np.mean([len(w) for w in words_lower]) if words_lower else 0
    avg_sent_len = n_words / n_sentences
    unique_words = set(words_lower)
    ttr = len(unique_words) / n_words_clean
    word_freq = Counter(words_lower)
    hapax = sum(1 for w, c in word_freq.items() if c == 1)
    hapax_ratio = hapax / n_words_clean
    func_count = sum(1 for w in words_lower if w in FUNCTION_WORDS)
    func_ratio = func_count / n_words_clean
    comma_rate = text.count(',') / n_words
    semicolon_rate = text.count(';') / n_words
    exclamation_rate = text.count('!') / n_words
    question_rate = text.count('?') / n_words
    upper_ratio = sum(1 for c in text if c.isupper()) / max(chars, 1)
    digit_ratio = sum(1 for c in text if c.isdigit()) / max(chars, 1)
    contractions = len(re.findall(r"\b\w+'\w+\b", text))
    contraction_rate = contractions / n_words
    newline_rate = text.count('\n') / max(chars, 1)
    freq_spectrum = Counter(word_freq.values())
    M = sum(i * i * freq_spectrum[i] for i in freq_spectrum)
    yules_k = 10000 * (M - n_words_clean) / (n_words_clean * n_words_clean) if n_words_clean > 1 else 0
    short_word_ratio = sum(1 for w in words_lower if len(w) <= 3) / n_words_clean
    long_word_ratio = sum(1 for w in words_lower if len(w) >= 7) / n_words_clean

    return np.array([
        avg_word_len / 10.0, avg_sent_len / 50.0, ttr, hapax_ratio,
        func_ratio, comma_rate, semicolon_rate, exclamation_rate,
        question_rate, upper_ratio, digit_ratio, contraction_rate,
        newline_rate, min(yules_k / 200.0, 1.0), short_word_ratio, long_word_ratio,
    ], dtype=np.float32)


STYLE_FEATURE_NAMES = [
    "avg_word_len", "avg_sent_len", "type_token_ratio", "hapax_ratio",
    "function_word_ratio", "comma_rate", "semicolon_rate", "exclamation_rate",
    "question_rate", "uppercase_ratio", "digit_ratio", "contraction_rate",
    "newline_rate", "yules_k", "short_word_ratio", "long_word_ratio",
]

print(f"Sample features: {extract_stylometric_features(train_df['text_1'].iloc[0])}")

Sample features: [0.39166668 0.15       1.         1.         0.5        0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.5        0.16666667]


## 4. Normalized Compression Distance (NCD)

Information-theoretic similarity measure from Jiang et al. (ACL 2023).

**Intuition:** If two texts share stylistic patterns (same author), their
concatenation will compress more efficiently than two unrelated texts because
the compressor can exploit shared character-level patterns, repetition habits,
and structural regularities.

$$NCD(x, y) = \frac{C(xy) - \min(C(x), C(y))}{\max(C(x), C(y))}$$

where $C(x)$ = compressed length of $x$.

- NCD ≈ 0: very similar (same author likely)
- NCD ≈ 1: very different (different author likely)

We also compute NCD in both directions (x+y and y+x) and use multiple
compression levels for richer features.

In [6]:
def compressed_len(text, level=9):
    """Get compressed length of text using gzip."""
    return len(gzip.compress(text.encode("utf-8"), compresslevel=level))


def compute_ncd(text_1, text_2, level=9):
    """Compute Normalized Compression Distance between two texts."""
    c1 = compressed_len(text_1, level)
    c2 = compressed_len(text_2, level)
    c12 = compressed_len(text_1 + " " + text_2, level)
    c21 = compressed_len(text_2 + " " + text_1, level)

    ncd_12 = (c12 - min(c1, c2)) / max(c1, c2)
    ncd_21 = (c21 - min(c1, c2)) / max(c1, c2)

    return ncd_12, ncd_21, c1, c2, c12


def compute_compression_features(df):
    """
    Compute compression-based features for all text pairs.
    Returns 6 features per pair:
    - NCD (text1+text2 direction)
    - NCD (text2+text1 direction)
    - Average NCD
    - Compression ratio of text_1
    - Compression ratio of text_2
    - Ratio of compressed lengths (symmetry indicator)
    """
    features = []
    for i in range(len(df)):
        t1 = df["text_1"].iloc[i]
        t2 = df["text_2"].iloc[i]

        ncd_12, ncd_21, c1, c2, c12 = compute_ncd(t1, t2)

        # Compression ratio = compressed / original (lower = more compressible)
        cr1 = c1 / max(len(t1.encode("utf-8")), 1)
        cr2 = c2 / max(len(t2.encode("utf-8")), 1)

        # Ratio of compressed lengths (how similar in compressibility)
        cr_ratio = min(c1, c2) / max(c1, c2) if max(c1, c2) > 0 else 1.0

        features.append([
            ncd_12,
            ncd_21,
            (ncd_12 + ncd_21) / 2.0,   # avg NCD
            cr1,
            cr2,
            cr_ratio,
        ])

    return np.array(features, dtype=np.float32)


# Quick test
same_author = compute_ncd("I love writing stories about cats!", "My cat is the best pet ever!")
diff_text = compute_ncd("I love writing stories about cats!", "The fiscal policy implications of monetary expansion are significant.")
print(f"Similar style NCD:  {same_author[0]:.4f}")
print(f"Different style NCD: {diff_text[0]:.4f}")
print(f"(Lower = more similar)")

Similar style NCD:  0.5370
Different style NCD: 0.6220
(Lower = more similar)


## 5. Full Feature Engineering Pipeline

In [7]:
 def build_features(df, char_tfidf=None, word_tfidf=None, scaler=None, fit=False):
    start = time.time()

    texts_1 = df["text_1"].tolist()
    texts_2 = df["text_2"].tolist()
    all_texts = texts_1 + texts_2

    # --- 1. Character TF-IDF ---
    print("  Building char TF-IDF...")
    if fit:
        char_tfidf = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5),
            max_features=10000, sublinear_tf=True,
        )
        char_tfidf.fit(all_texts)

    char_1 = char_tfidf.transform(texts_1)
    char_2 = char_tfidf.transform(texts_2)

    # --- 2. Word TF-IDF ---
    print("  Building word TF-IDF...")
    if fit:
        word_tfidf = TfidfVectorizer(
            analyzer="word", ngram_range=(1, 2),
            max_features=10000, sublinear_tf=True,
        )
        word_tfidf.fit(all_texts)

    word_1 = word_tfidf.transform(texts_1)
    word_2 = word_tfidf.transform(texts_2)

    # --- 3. Cosine similarities ---
    print("  Computing cosine similarities...")
    char_cos = np.array([cosine_similarity(char_1[i], char_2[i])[0,0] for i in range(len(df))]).reshape(-1,1)
    word_cos = np.array([cosine_similarity(word_1[i], word_2[i])[0,0] for i in range(len(df))]).reshape(-1,1)

    # --- 4. TF-IDF differences ---
    print("  Computing TF-IDF differences...")
    char_diff = abs(char_1 - char_2)
    word_diff = abs(word_1 - word_2)

    # --- 5. Stylometric features ---
    print("  Extracting stylometric features...")
    style_1 = np.array([extract_stylometric_features(t) for t in texts_1])
    style_2 = np.array([extract_stylometric_features(t) for t in texts_2])
    style_abs_diff = np.abs(style_1 - style_2)
    style_product = style_1 * style_2

    # --- 6. Compression features ---
    print("  Computing NCD...")
    compression_features = compute_compression_features(df)

    # --- 7. Combine dense ---
    dense_features = np.hstack([
        char_cos, word_cos,
        style_1, style_2, style_abs_diff, style_product,
        compression_features,
    ])

    if fit:
        scaler = StandardScaler()
        dense_features = scaler.fit_transform(dense_features)
    else:
        dense_features = scaler.transform(dense_features)

    # --- 8. Combine all ---
    feature_matrix = hstack([char_diff, word_diff, csr_matrix(dense_features)])

    elapsed = time.time() - start
    print(f"  Done. Shape: {feature_matrix.shape} ({elapsed:.1f}s)")
    return feature_matrix, char_tfidf, word_tfidf, scaler

In [8]:
print("=== Building TRAIN features ===")
X_train, char_vec, word_vec, scaler = build_features(train_df, fit=True)
y_train = train_df["label"].values

print("\n=== Building DEV features ===")
X_dev, _, _, _ = build_features(dev_df, char_tfidf=char_vec, word_tfidf=word_vec, scaler=scaler, fit=False)
y_dev = dev_df["label"].values

print(f"\nTrain: {X_train.shape} | Dev: {X_dev.shape}")

=== Building TRAIN features ===
  Building char TF-IDF...
  Building word TF-IDF...
  Computing cosine similarities...
  Computing TF-IDF differences...
  Extracting stylometric features...
  Computing NCD...
  Done. Shape: (27643, 20072) (292.0s)

=== Building DEV features ===
  Building char TF-IDF...
  Building word TF-IDF...
  Computing cosine similarities...
  Computing TF-IDF differences...
  Extracting stylometric features...
  Computing NCD...
  Done. Shape: (5993, 20072) (19.7s)

Train: (27643, 20072) | Dev: (5993, 20072)


## 6. Train Classifiers

In [9]:
print("Training Logistic Regression...")
lr_start = time.time()
lr_model = LogisticRegression(
    C=5.0,              # was 1.0 — stronger regularisation often helps
    max_iter=2000,
    solver="lbfgs",
    random_state=CONFIG["seed"],
)
lr_model.fit(X_train, y_train)
lr_time = time.time() - lr_start

lr_preds = lr_model.predict(X_dev)
lr_probs = lr_model.predict_proba(X_dev)[:, 1]
lr_f1 = f1_score(y_dev, lr_preds, average="macro")
print(f"  LR Dev F1: {lr_f1:.4f} ({lr_time:.1f}s)")

Training Logistic Regression...
  LR Dev F1: 0.6905 (103.1s)


In [10]:
print("Training SGD Classifier...")
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV

sgd_start = time.time()
base_sgd = SGDClassifier(
    loss="modified_huber",    # gives better probabilities than log_loss
    alpha=0.00005,            # less regularisation
    max_iter=2000,
    random_state=CONFIG["seed"],
)
sgd_model = CalibratedClassifierCV(base_sgd, cv=3)
sgd_model.fit(X_train, y_train)
sgd_time = time.time() - sgd_start

sgd_preds = sgd_model.predict(X_dev)
sgd_probs = sgd_model.predict_proba(X_dev)[:, 1]
sgd_f1 = f1_score(y_dev, sgd_preds, average="macro")
print(f"  SGD Dev F1: {sgd_f1:.4f} ({sgd_time:.1f}s)")

Training SGD Classifier...
  SGD Dev F1: 0.6774 (9.7s)


In [11]:
print(f"Logistic Regression: F1 = {lr_f1:.4f}")
print(f"Random Forest:       F1 = {sgd_f1:.4f}")

Logistic Regression: F1 = 0.6905
Random Forest:       F1 = 0.6774


## 7. Ensemble

In [12]:
ensemble_probs = (lr_probs + sgd_probs) / 2.0
ensemble_preds = (ensemble_probs >= 0.5).astype(int)
ensemble_f1 = f1_score(y_dev, ensemble_preds, average="macro")

print("=" * 60)
print("  ENSEMBLE RESULTS ON DEV SET")
print("=" * 60)
print(f"\nLR F1:       {lr_f1:.4f}")
print(f"SVM F1:      {sgd_f1:.4f}")
print(f"Ensemble F1: {ensemble_f1:.4f}")
print(f"\n{classification_report(y_dev, ensemble_preds, digits=4)}")
print("Confusion Matrix:")
print(confusion_matrix(y_dev, ensemble_preds))

  ENSEMBLE RESULTS ON DEV SET

LR F1:       0.6905
SVM F1:      0.6774
Ensemble F1: 0.6928

              precision    recall  f1-score   support

           0     0.6829    0.6966    0.6897      2937
           1     0.7027    0.6891    0.6959      3056

    accuracy                         0.6928      5993
   macro avg     0.6928    0.6929    0.6928      5993
weighted avg     0.6930    0.6928    0.6928      5993

Confusion Matrix:
[[2046  891]
 [ 950 2106]]


## 8. Threshold Optimisation

In [13]:
best_thresh = 0.5
best_f1 = ensemble_f1

for thresh in np.arange(0.30, 0.70, 0.01):
    preds = (ensemble_probs >= thresh).astype(int)
    f1 = f1_score(y_dev, preds, average="macro")
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = thresh

print(f"Default (0.50): F1 = {ensemble_f1:.4f}")
print(f"Optimal ({best_thresh:.2f}):  F1 = {best_f1:.4f}")

final_preds = (ensemble_probs >= best_thresh).astype(int)
print(f"\n{classification_report(y_dev, final_preds, digits=4)}")

Default (0.50): F1 = 0.6928
Optimal (0.50):  F1 = 0.6928

              precision    recall  f1-score   support

           0     0.6829    0.6966    0.6897      2937
           1     0.7027    0.6891    0.6959      3056

    accuracy                         0.6928      5993
   macro avg     0.6928    0.6929    0.6928      5993
weighted avg     0.6930    0.6928    0.6928      5993



## 9. Feature Importance

In [18]:
# Feature importance — offset accounts for BOTH char + word TF-IDF diff vectors
n_sparse = 10000 + 10000  # char_diff (10K) + word_diff (10K)
dense_coefs = lr_model.coef_[0][n_sparse:]

dense_names = (
    ["char_cosine_sim", "word_cosine_sim"] +
    [f"t1_{n}" for n in STYLE_FEATURE_NAMES] +
    [f"t2_{n}" for n in STYLE_FEATURE_NAMES] +
    [f"diff_{n}" for n in STYLE_FEATURE_NAMES] +
    [f"prod_{n}" for n in STYLE_FEATURE_NAMES] +
    ["ncd_12", "ncd_21", "ncd_avg", "compress_ratio_1", "compress_ratio_2", "compress_ratio_sym"]
)

importance = sorted(zip(dense_names, dense_coefs), key=lambda x: abs(x[1]), reverse=True)

print("Top 20 most important features (by LR coefficient magnitude):")
print("-" * 55)
for name, coef in importance[:20]:
    direction = "→ same author" if coef > 0 else "→ diff author"
    print(f"  {name:35s} {coef:+.4f}  {direction}")

Top 20 most important features (by LR coefficient magnitude):
-------------------------------------------------------
  diff_exclamation_rate               -2.3710  → diff author
  t1_exclamation_rate                 +1.8758  → same author
  t2_exclamation_rate                 +1.5880  → same author
  t2_question_rate                    +1.2030  → same author
  diff_question_rate                  -1.1723  → diff author
  word_cosine_sim                     +1.0808  → same author
  diff_avg_word_len                   -1.0296  → diff author
  diff_semicolon_rate                 -0.9836  → diff author
  prod_hapax_ratio                    -0.8815  → diff author
  t2_hapax_ratio                      +0.8488  → same author
  t2_semicolon_rate                   +0.8093  → same author
  t1_avg_word_len                     +0.7973  → same author
  char_cosine_sim                     +0.6897  → same author
  t2_uppercase_ratio                  +0.6562  → same author
  t1_uppercase_ratio        

## 10. Error Analysis

In [19]:
dev_df_analysis = dev_df.copy()
dev_df_analysis["pred"] = final_preds
dev_df_analysis["prob"] = ensemble_probs
dev_df_analysis["correct"] = (dev_df_analysis["pred"] == dev_df_analysis["label"]).astype(int)

errors = dev_df_analysis[dev_df_analysis["correct"] == 0]
print(f"Errors: {len(errors)} / {len(dev_df_analysis)} ({len(errors)/len(dev_df_analysis)*100:.1f}%)")
print(f"By true label: {errors['label'].value_counts().to_dict()}")

Errors: 1841 / 5993 (30.7%)
By true label: {1: 950, 0: 891}


In [20]:
print("=== Confident FALSE POSITIVES ===")
fp = errors[errors["label"] == 0].nlargest(3, "prob")
for i, (_, row) in enumerate(fp.iterrows(), 1):
    print(f"\n  [{i}] Prob: {row['prob']:.3f}")
    print(f"  T1: {row['text_1'][:120]}...")
    print(f"  T2: {row['text_2'][:120]}...")

print("\n=== Confident FALSE NEGATIVES ===")
fn = errors[errors["label"] == 1].nsmallest(3, "prob")
for i, (_, row) in enumerate(fn.iterrows(), 1):
    print(f"\n  [{i}] Prob: {row['prob']:.3f}")
    print(f"  T1: {row['text_1'][:120]}...")
    print(f"  T2: {row['text_2'][:120]}...")

=== Confident FALSE POSITIVES ===

  [1] Prob: 0.915
  T1: Faith, Attached is information regarding the Texas Desk as it relates to the current Texas market, EOL performance metri...
  T2: At the request of Jake Staffel, I am enclosing our proposed form of Non-Disclosure Agreement. If you have any comments o...

  [2] Prob: 0.907
  T1: This is one of the most accomplished films I've seen from France in a while . French cinema always presents risky situat...
  T2: Shinya Tsukamoto , the man behind the Tetsuo films , Snake of June and Tokyo Fist takes on Edogawa Rampo story and turns...

  [3] Prob: 0.904
  T1: Start Date: 4/27/01; HourAhead hour: 5; No ancillary schedules awarded. No variances detected. LOG MESSAGES: PARSING FIL...
  T2: More for the contract... ---------------------- Forwarded by Kay Mann/Corp/Enron on 02/15/2001 03:26 PM ----------------...

=== Confident FALSE NEGATIVES ===

  [1] Prob: 0.062
  T1: Thank you. Kay...
  T2: You are difficult to keep up with!...

  [2] 

In [21]:
with open(os.path.join(CONFIG["output_dir"], "lr_model.pkl"), "wb") as f:
    pickle.dump(lr_model, f)
with open(os.path.join(CONFIG["output_dir"], "sgd_model.pkl"), "wb") as f:
    pickle.dump(sgd_model, f)
with open(os.path.join(CONFIG["output_dir"], "char_tfidf.pkl"), "wb") as f:
    pickle.dump(char_vec, f)
with open(os.path.join(CONFIG["output_dir"], "word_tfidf.pkl"), "wb") as f:
    pickle.dump(word_vec, f)
with open(os.path.join(CONFIG["output_dir"], "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print("All models saved.")

All models saved.


## 12. Inference — Test Data (Demo Code)

In [22]:
def run_inference(test_path, output_path, threshold=0.5):
    with open(os.path.join(CONFIG["output_dir"], "lr_model.pkl"), "rb") as f:
        lr = pickle.load(f)
    with open(os.path.join(CONFIG["output_dir"], "sgd_model.pkl"), "rb") as f:
        sgd = pickle.load(f)
    with open(os.path.join(CONFIG["output_dir"], "char_tfidf.pkl"), "rb") as f:
        char_tf = pickle.load(f)
    with open(os.path.join(CONFIG["output_dir"], "word_tfidf.pkl"), "rb") as f:
        word_tf = pickle.load(f)
    with open(os.path.join(CONFIG["output_dir"], "scaler.pkl"), "rb") as f:
        sc = pickle.load(f)

    test_df = load_data(test_path)
    print("Building test features...")
    X_test, _, _, _ = build_features(
        test_df, char_tfidf=char_tf, word_tfidf=word_tf, scaler=sc, fit=False
    )

    lr_probs = lr.predict_proba(X_test)[:, 1]
    sgd_probs = sgd.predict_proba(X_test)[:, 1]
    ensemble_probs = (lr_probs + sgd_probs) / 2.0
    preds = (ensemble_probs >= threshold).astype(int)

    pd.DataFrame({"prediction": preds}).to_csv(output_path, index=False)
    print(f"Saved to {output_path}")
    print(f"Distribution: {pd.Series(preds).value_counts().to_dict()}")
    return preds

In [23]:
# UNCOMMENT WHEN TEST DATA IS AVAILABLE
test_preds = run_inference(
    test_path="/content/dev.csv",
    output_path="/content/Group_31_A.csv",
    threshold=best_thresh,
)

Building test features...
  Building char TF-IDF...
  Building word TF-IDF...
  Computing cosine similarities...
  Computing TF-IDF differences...
  Extracting stylometric features...
  Computing NCD...
  Done. Shape: (5993, 20072) (46.3s)
Saved to /content/Group_31_A.csv
Distribution: {1: 2997, 0: 2996}


## 13. Save Summary

In [24]:
summary = {
    "approach": "A — Traditional ML (Char+Word TF-IDF + Stylometric + NCD + LR/SGD Ensemble)",
    "creative_elements": [
        "TF-IDF character n-grams (3-5) for subword stylistic patterns",
        "TF-IDF word n-grams (1-2) for lexical patterns",
        "16 handcrafted stylometric features per text",
        "Normalized Compression Distance (NCD) — information-theoretic similarity (Jiang et al., ACL 2023)",
        "Compression ratio and symmetry features",
        "Pairwise difference features |style_1 - style_2|",
        "Pairwise product features style_1 * style_2",
        "Dual TF-IDF cosine similarity (char + word level)",
        "TF-IDF absolute difference vectors (char + word)",
        "LR + SGD Classifier ensemble",
        "Threshold optimisation on dev set",
    ],
    "references": [
        "Jiang et al. (2023) Low-Resource Text Classification: A Parameter-Free Classification Method with Compressors, ACL 2023",
    ],
    "config": {k: str(v) for k, v in CONFIG.items()},
    "results": {
        "lr_dev_f1": round(float(lr_f1), 4),
        "sgd_dev_f1": round(float(sgd_f1), 4),
        "ensemble_dev_f1": round(float(ensemble_f1), 4),
        "optimised_dev_f1": round(float(best_f1), 4),
        "optimal_threshold": round(float(best_thresh), 2),
    },
}

with open(os.path.join(CONFIG["output_dir"], "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nDone!")

{
  "approach": "A \u2014 Traditional ML (Char+Word TF-IDF + Stylometric + NCD + LR/SGD Ensemble)",
  "creative_elements": [
    "TF-IDF character n-grams (3-5) for subword stylistic patterns",
    "TF-IDF word n-grams (1-2) for lexical patterns",
    "16 handcrafted stylometric features per text",
    "Normalized Compression Distance (NCD) \u2014 information-theoretic similarity (Jiang et al., ACL 2023)",
    "Compression ratio and symmetry features",
    "Pairwise difference features |style_1 - style_2|",
    "Pairwise product features style_1 * style_2",
    "Dual TF-IDF cosine similarity (char + word level)",
    "TF-IDF absolute difference vectors (char + word)",
    "LR + SGD Classifier ensemble",
    "Threshold optimisation on dev set"
  ],
  "references": [
    "Jiang et al. (2023) Low-Resource Text Classification: A Parameter-Free Classification Method with Compressors, ACL 2023"
  ],
  "config": {
    "seed": "42",
    "tfidf_max_features": "50000",
    "tfidf_ngram_range": "